In [1]:
import pandas as pd

from src.app.utils.db_manager import DatabaseManager

In [2]:
from warnings import filterwarnings

filterwarnings('ignore', category=UserWarning, message='.*pandas only supports SQLAlchemy connectable.*')

pd.options.display.float_format = '{:,.2f}'.format

In [3]:
db_manager = DatabaseManager()

## Выполнение SQL-запросов

### Вывести средний доход среди всех клиентов

In [4]:
sql_query = """
    SELECT AVG(amt_income_total) AS average_income
    FROM application;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,average_income
0,"170,116.06"


### Вывести минимальный и максимальный возраст среди всех клиентов

В таблице в days_birth хранится количество дней с отрицательным знаком, нужно перевести в года

In [5]:
sql_query = """
    SELECT
        MIN(ABS(days_birth) / 365) AS min_age,
        MAX(ABS(days_birth) / 365) AS max_age
    FROM application;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,min_age,max_age
0,20,69


### Вывести количество мужчин и женщин

In [6]:
sql_query = """
    SELECT
        code_gender,
        COUNT(*) AS count
    FROM application
    GROUP BY code_gender;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,code_gender,count
0,F,235126
1,M,121125
2,XNA,4


### Вывести общую сумму, количество и среднюю сумму, запрошенную клиентами в кредит с авто и без

In [7]:
sql_query = """
    SELECT
        flag_own_car,
        SUM(amt_credit) AS total_amount_credit,
        COUNT(*) AS total_count,
        AVG(amt_credit) AS average_amount_credit
    FROM application
    GROUP BY flag_own_car;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,flag_own_car,total_amount_credit,total_count,average_amount_credit
0,N,"130,391,450,000.00",235235,"554,317.38"
1,Y,"79,004,420,000.00",121020,"652,786.57"


### Вывести доли клиентов с различным образованием

In [8]:
sql_query = """
    WITH total AS (
        SELECT COUNT(*) AS count FROM application
    )
    SELECT
        name_education_type,
        COUNT(*) * 100.0 / total.count  AS percentage
    FROM application, total
    GROUP BY name_education_type, total.count;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,name_education_type,percentage
0,Higher education,24.53
1,Secondary / secondary special,70.84
2,Lower secondary,1.20
3,Incomplete higher,3.37
4,Academic degree,0.06


### Подсчитать количество полных лет для клиентов, у которых есть во владении автомобиль и недвижимость, вывести топ 10 по возрастанию

In [9]:
sql_query = """
    SELECT
        sk_id_curr,
        ABS(days_birth) / 365 AS age,
        flag_own_car,
        flag_own_realty
    FROM application
    WHERE flag_own_car = 'Y' and flag_own_realty = 'Y'
    ORDER BY age
    LIMIT 10;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,sk_id_curr,age,flag_own_car,flag_own_realty
0,372716,20,Y,Y
1,283094,20,Y,Y
2,125940,21,Y,Y
3,118956,21,Y,Y
4,103397,21,Y,Y
5,123826,21,Y,Y
6,107794,21,Y,Y
7,107188,21,Y,Y
8,102955,21,Y,Y
9,110334,21,Y,Y


### Вывести тех клиентов, у кого доход на одного члена семьи в два раза больше, чем в среднем на одного члена семьи по выборке

In [10]:
sql_query = """
    WITH income_per_family_member AS (
        SELECT AVG(amt_income_total / cnt_fam_members) AS average_income_per_family_member FROM application
    )
    SELECT
        sk_id_curr,
        amt_income_total,
        cnt_fam_members,
        amt_income_total / cnt_fam_members AS income_per_family_member,
        average_income_per_family_member
    FROM application, income_per_family_member AS i
    GROUP BY sk_id_curr, amt_income_total, cnt_fam_members, i.average_income_per_family_member
    HAVING amt_income_total / cnt_fam_members > 2 * average_income_per_family_member;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,sk_id_curr,amt_income_total,cnt_fam_members,income_per_family_member,average_income_per_family_member
0,292656,"450,000.00",2.00,"225,000.00","93,786.30"
1,224661,"247,500.00",1.00,"247,500.00","93,786.30"
2,171004,"225,000.00",1.00,"225,000.00","93,786.30"
3,139042,"193,500.00",1.00,"193,500.00","93,786.30"
4,288029,"202,500.00",1.00,"202,500.00","93,786.30"
...,...,...,...,...,...
27728,253663,"315,000.00",1.00,"315,000.00","93,786.30"
27729,363931,"315,000.00",1.00,"315,000.00","93,786.30"
27730,286445,"225,000.00",1.00,"225,000.00","93,786.30"
27731,233934,"540,000.00",1.00,"540,000.00","93,786.30"


### Вывести клиентов старше 60 лет по которым нет данных в bureau

In [11]:
sql_query = """
    SELECT
        a.sk_id_curr,
        ABS(a.days_birth) / 365 AS age,
        b.sk_id_curr
    FROM application a
    LEFT JOIN bureau b ON a.sk_id_curr = b.sk_id_curr
    WHERE ABS(a.days_birth) / 365 > 60 AND b.sk_id_curr IS NULL;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,sk_id_curr,age,sk_id_curr
0,100118,61,None
1,100236,63,None
2,100273,63,None
3,100287,68,None
4,100413,62,None
...,...,...,...
5051,444622,64,None
5052,445713,63,None
5053,450555,63,None
5054,451383,66,None


### Вывести женщин, у которых в истории bureau было больше двух кредитов, просроченных на 61 день и более, отсортировать в порядке убывания по кол-ву таких кредитов

In [12]:
sql_query = """
    SELECT
        a.sk_id_curr,
        COUNT(b.sk_id_bureau) AS overdue_credits_count
    FROM application a
    JOIN bureau b ON a.sk_id_curr = b.sk_id_curr
    WHERE a.code_gender = 'F' AND b.credit_day_overdue >= 61
    GROUP BY a.sk_id_curr
    HAVING COUNT(b.sk_id_bureau) > 2
    ORDER BY overdue_credits_count DESC;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,sk_id_curr,overdue_credits_count
0,264144,5
1,142384,4
2,374345,3
3,375724,3
4,431820,3
5,114166,3
6,436084,3
7,337741,3


### По данным из bureau (БКИ) рассчитать долю просрочки в активных займах для каждого клиента, вывести топ 7 мужчин с наибольшей суммой просрочки, указав для них, помимо прочего, сумму активных кредитов и суммы всех кредитов (активных и закрытых).

In [13]:
sql_query = """
    WITH male_bureau_stats AS (
        SELECT 
            a.sk_id_curr,
            SUM(CASE WHEN b.credit_active = 'Active' THEN b.amt_credit_sum_overdue ELSE 0 END) AS active_overdue_sum,
            SUM(CASE WHEN b.credit_active = 'Active' THEN b.amt_credit_sum ELSE 0 END) AS active_credits_sum,
            SUM(b.amt_credit_sum) AS all_credits_total
        FROM application a
        JOIN bureau b ON a.sk_id_curr = b.sk_id_curr
        WHERE a.code_gender = 'M'
        GROUP BY a.sk_id_curr
    )
    SELECT 
        *,
        active_overdue_sum / NULLIF(active_credits_sum, 0) AS active_overdue_percentage
    FROM male_bureau_stats
    ORDER BY active_overdue_sum DESC
    LIMIT 7;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,sk_id_curr,active_overdue_sum,active_credits_sum,all_credits_total,active_overdue_percentage
0,435405,"3,681,063.00","3,690,000.00","4,054,000.50",1.00
1,427996,"1,571,697.00","4,231,615.50","7,431,583.50",0.37
2,394113,"1,332,472.50","1,530,000.00","1,825,800.90",0.87
3,266765,"1,224,474.90","1,350,000.00","1,421,955.00",0.91
4,167085,"780,192.00","158,404.50","176,404.50",4.93
5,154595,"742,491.00","990,000.00","4,245,754.50",0.75
6,262411,"709,669.25","1,318,500.00","1,773,000.00",0.54


## Новые признаки

### 1. Возраст

Для удобства, можно вывести возраст каждого клиента в годах

In [14]:
sql_query = """
    SELECT
        sk_id_curr,
        ABS(days_birth) / 365 AS age
    FROM application;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,sk_id_curr,age
0,100002,25
1,100003,45
2,100004,52
3,100006,52
4,100007,54
...,...,...
356250,456221,54
356251,456222,30
356252,456223,43
356253,456224,38


### 2. Общая сумма текущей задолженности клиента во всех банках

Сумма AMT_CREDIT_SUM_DEBT для всех активных записей. Если у клиента уже много открытых кредитов, риск дефолта намного выше

In [15]:
sql_query = """
    SELECT 
        a.sk_id_curr,
        SUM(b.amt_credit_sum_debt) AS total_bureau_debt
    FROM application a
    LEFT JOIN bureau b ON a.sk_id_curr = b.sk_id_curr
    WHERE b.credit_active = 'Active' OR b.credit_active IS NULL
    GROUP BY a.sk_id_curr;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:01


,sk_id_curr,total_bureau_debt
0,233338,"180,103.50"
1,129976,"6,161,193.00"
2,243212,NaN
3,329035,"1,008,233.00"
4,157514,"194,931.94"
...,...,...
302254,158829,"1,098,783.00"
302255,290047,NaN
302256,279693,"175,072.50"
302257,106095,"430,083.00"


### 3. Отношение кредита к стоимости товара

Какую часть стоимости товара клиент покрывает кредитом. Если клиент берет в кредит 100% стоимости или больше, у него явно нет накоплений. Если он внес большой первоначальный взнос, он более надежен

In [16]:
sql_query = """
    SELECT 
        sk_id_curr,
        amt_credit / NULLIF(amt_goods_price, 0) AS credit_to_goods
    FROM application
    WHERE amt_goods_price IS NOT NULL
    ORDER BY credit_to_goods;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,sk_id_curr,credit_to_goods
0,298406,0.15
1,344802,0.15
2,333948,0.23
3,335414,0.23
4,441153,0.25
...,...,...
355972,362717,4.67
355973,187078,4.67
355974,379070,5.00
355975,146687,6.00


### 4. Остаток после ежемесячной выплаты

Сколько денег остается у клиента на жизнь после выплаты ежемесячного взноса

In [17]:
sql_query = """
    SELECT
        sk_id_curr,
        amt_income_total,
        amt_annuity,
        amt_income_total - amt_annuity AS income_after_annuity
    FROM application;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,sk_id_curr,amt_income_total,amt_annuity,income_after_annuity
0,100002,"202,500.00","24,700.50","177,799.50"
1,100003,"270,000.00","35,698.50","234,301.50"
2,100004,"67,500.00","6,750.00","60,750.00"
3,100006,"135,000.00","29,686.50","105,313.50"
4,100007,"121,500.00","21,865.50","99,634.50"
...,...,...,...,...
356250,456221,"121,500.00","17,473.50","104,026.50"
356251,456222,"157,500.00","31,909.50","125,590.50"
356252,456223,"202,500.00","33,205.50","169,294.50"
356253,456224,"225,000.00","25,128.00","199,872.00"


### 5. Количество лет работы на текущем месте

Какую часть трудоспособного возраста (с 18 лет) человек провел на текущем месте работы. Показывает стабильность клиента

В данных значение days_employed для безработных или пенсионеров - очень большое. Нужно это учесть, заменив на 0

In [18]:
sql_query = """
    SELECT 
        sk_id_curr,
        days_employed,
        days_birth,
        (CASE WHEN days_employed > 0 THEN 0 ELSE ABS(days_employed) END) * 1.0 * 1.0 / (ABS(days_birth) - (18 * 365)) AS work_to_life
    FROM application;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:01


,sk_id_curr,days_employed,days_birth,work_to_life
0,100002,-637,-9461,0.22
1,100003,-1188,-16765,0.12
2,100004,-225,-19046,0.02
3,100006,-3039,-19005,0.24
4,100007,-3038,-19932,0.23
...,...,...,...,...
356250,456221,-5169,-19970,0.39
356251,456222,-1149,-11186,0.25
356252,456223,-3037,-15922,0.32
356253,456224,-2731,-13968,0.37


### 6. Среднее количество дней просрочки по прошлым кредитам в этом банке

Разница между датой фактического платежа и датой по графику. Показывает ответственность клиента

In [19]:
sql_query = """
    SELECT 
        sk_id_curr,
        AVG(CASE WHEN days_entry_payment > days_instalment THEN days_entry_payment - days_instalment ELSE 0 END) AS average_days_late
    FROM installments_payments
    GROUP BY sk_id_curr;
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:05


,sk_id_curr,average_days_late
0,100001,1.57
1,100002,0.00
2,100003,0.00
3,100004,0.00
4,100005,0.11
...,...,...
339582,456251,0.00
339583,456252,0.50
339584,456253,0.64
339585,456254,0.00
